In [24]:
import os
import shutil
import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from collections import Counter
from IPython.display import Audio, display

In [30]:
CONFIG = {
    "raw_dir"    : "/content/icbhi_dataset/raw/Respiratory_Sound_Database/Respiratory_Sound_Database",      # downloaded .wav + .txt
    "cycles_dir" : "/content/icbhi_dataset/cycles",   # segmented cycles go here
    "splits_file": "/content/icbhi_dataset/raw/ICBHI_challenge_train_test.txt",


    "target_sr"  : 22050,

    "class_map" : {
        (0,0): "Normal",
        (1,0): "Crackle",
        (0,1): "Wheeze",
        (1,1): "Both",
    },
    "classes"    : ["Normal", "Crackle", "Wheeze", "Both"],

    "seed" : 42,

}

In [45]:
master_df = build_master_annotation(CONFIG["raw_dir"], CONFIG["splits_file"])


--- Split file content (first 5 rows):
                                             filename  split
0  <!DOCTYPE html><html><head><title>404 Not Foun...    NaN

Found 920 annotation files...
  Debug: Stem '101_1b1_Al_sc_Meditron' resolved to split 'unknown'
  Debug: Stem '101_1b1_Pr_sc_Meditron' resolved to split 'unknown'
  Debug: Stem '102_1b1_Ar_sc_Meditron' resolved to split 'unknown'
  Debug: Stem '103_2b2_Ar_mc_LittC2SE' resolved to split 'unknown'
  Debug: Stem '104_1b1_Al_sc_Litt3200' resolved to split 'unknown'

Master DataFrame built.
Total breathing cycles : 6898
Unique patients        : 126
Train cycles           : 0
Test cycles            : 0


In [33]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d vbookshelf/respiratory-sound-database \
       -p /content/icbhi_dataset/raw --unzip --quiet

print("Download complete.")

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/vbookshelf/respiratory-sound-database
License(s): unknown
Download complete.


In [48]:
def parse_annotation_file(txt_path: str) -> pd.DataFrame:
  """
    Read one ICBHI annotation .txt file and return a DataFrame.

    Args:
        txt_path: path to the .txt annotation file

    Returns:
        DataFrame with columns:
            start    → cycle start time in seconds (float)
            end      → cycle end time in seconds (float)
            crackle  → 1 if crackles present, else 0 (int)
            wheeze   → 1 if wheeze present, else 0 (int)
            label    → human-readable class string ("Normal", "Crackle", etc.)
    """

  df = pd.read_csv(
        txt_path,
        sep="\t",
        header=None,
        names=["start", "end", "crackle", "wheeze"],
    )
  df["crackle"] = df["crackle"].astype(int)
  df["wheeze"] = df["wheeze"].astype(int)


  df["label"] = df.apply(
        lambda row: CONFIG["class_map"][(row["crackle"], row["wheeze"])],
        axis=1
    )

  return df

In [49]:
def demo_annotation_parsing(raw_dir: str):
    """Show the first annotation file we find."""
    audio_dir = os.path.join(raw_dir, "audio_and_txt_files")

    # Find any .txt file
    txt_files = list(Path(audio_dir).glob("*.txt"))
    if not txt_files:
        print("No annotation files found. Check your raw_dir path.")
        return

    sample_txt = str(txt_files[0])
    print(f"Parsing: {os.path.basename(sample_txt)}\n")

    df = parse_annotation_file(sample_txt)
    print(df.to_string(index=False))
    print(f"\nCycles in this recording: {len(df)}")
    print(f"Label counts: {df['label'].value_counts().to_dict()}")

In [50]:
demo_annotation_parsing(CONFIG["raw_dir"])

Parsing: 158_1b3_Ar_mc_LittC2SE.txt

 start    end  crackle  wheeze   label
 0.079  2.007        0       0  Normal
 2.007  3.993        0       0  Normal
 3.993  5.764        0       0  Normal
 5.764  7.693        1       0 Crackle
 7.693  9.107        0       0  Normal
 9.107 10.850        1       0 Crackle
10.850 12.364        1       0 Crackle
12.364 15.021        1       0 Crackle
15.021 16.550        1       0 Crackle
16.550 18.479        1       0 Crackle
18.479 19.950        0       0  Normal

Cycles in this recording: 11
Label counts: {'Crackle': 6, 'Normal': 5}


In [53]:
def build_master_annotation(raw_dir: str, splits_file: str) -> pd.DataFrame:
  """
    Parse all annotation files and combine into one master DataFrame.

    Also attaches:
        - patient_id  → extracted from filename (first number, e.g. "101")
        - split       → "train" or "test" from official ICBHI split file
        - wav_path    → full path to the source .wav file
        - txt_path    → full path to the annotation .txt file

    Args:
        raw_dir     : path to the raw dataset root
        splits_file : path to ICBHI_challenge_train_test.txt

    Returns:
        master DataFrame with one row per breathing cycle
    """
  audio_dir = os.path.join(raw_dir, "audio_and_txt_files")
   # The split file has one filename per line, labeled "train" or "test"
  # Format:  101_1b1_Al_sc_Litt3200   train
  split_df = pd.read_csv(
      splits_file,
      sep="\t",
      header=None,
      names=["filename", "split"]
  )
  print("\n--- Split file content (first 5 rows):\n", split_df.head())

  # Build a dict: filename_stem → "train" or "test"
  split_dict = dict(zip(split_df["filename"], split_df["split"]))

  all_rows = []   # collect rows from every file

  txt_files = sorted(Path(audio_dir).glob("*.txt"))
  print(f"\nFound {len(txt_files)} annotation files...")

  # Limit to first few files for inspection
  _debug_count = 0
  for txt_path in txt_files:
      stem = txt_path.stem               # e.g. "101_1b1_Al_sc_Litt3200"
      wav_path = txt_path.with_suffix(".wav")  # matching .wav file

      if not wav_path.exists():
          # Skip if audio file missing (shouldn't happen in clean download)
          print(f"  WARNING: No .wav found for {stem}, skipping.")
          continue

      # Extract patient ID — always the first part before the first underscore
      patient_id = stem.split("_")[0]    # "101_1b1_..." → "101"

      # Get train/test label from the split dict
      split = split_dict.get(stem, "unknown")

      if _debug_count < 5: # Print first 5 stems and their resolved split
          print(f"  Debug: Stem '{stem}' resolved to split '{split}'")
          _debug_count += 1

      # Parse this file's annotation
      df = parse_annotation_file(str(txt_path))

      # Attach metadata columns to every row
      df["filename"]   = stem
      df["patient_id"] = patient_id
      df["split"]      = split
      df["wav_path"]   = str(wav_path)
      df["txt_path"]   = str(txt_path)

      all_rows.append(df)

  # Stack all individual DataFrames into one master table
  master = pd.concat(all_rows, ignore_index=True)

  # Add a unique cycle_id for each row (useful for saving files later)
  master["cycle_id"] = master.index

  print(f"\nMaster DataFrame built.")
  print(f"Total breathing cycles : {len(master)}")
  print(f"Unique patients        : {master['patient_id'].nunique()}")
  print(f"Train cycles           : {(master['split'] == 'train').sum()}")
  print(f"Test cycles            : {(master['split'] == 'test').sum()}")

  return master

In [43]:
master_df = build_master_annotation(CONFIG["raw_dir"], CONFIG["splits_file"])

Found 920 annotation files...

Master DataFrame built.
Total breathing cycles : 6898
Unique patients        : 126
Train cycles           : 0
Test cycles            : 0


In [39]:
main_folder = r"/content/icbhi_dataset"

for root, dirs, files in os.walk(main_folder):
    for folder in dirs:
        folder_path = os.path.join(root, folder)
        print(folder_path)

/content/icbhi_dataset/cycles
/content/icbhi_dataset/raw
/content/icbhi_dataset/raw/Respiratory_Sound_Database
/content/icbhi_dataset/raw/respiratory_sound_database
/content/icbhi_dataset/raw/Respiratory_Sound_Database/Respiratory_Sound_Database
/content/icbhi_dataset/raw/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files
/content/icbhi_dataset/raw/respiratory_sound_database/Respiratory_Sound_Database
/content/icbhi_dataset/raw/respiratory_sound_database/Respiratory_Sound_Database/audio_and_txt_files


In [40]:
import os

for root, dirs, files in os.walk("/content/icbhi_dataset"):
    if files:
        print(root)
        print("   ", files[:10])

/content/icbhi_dataset/raw
    ['demographic_info.txt']
/content/icbhi_dataset/raw/Respiratory_Sound_Database/Respiratory_Sound_Database
    ['filename_format.txt', 'patient_diagnosis.csv', 'filename_differences.txt']
/content/icbhi_dataset/raw/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files
    ['158_1b3_Ar_mc_LittC2SE.txt', '203_1p3_Tc_mc_AKGC417L.txt', '138_2p2_Al_mc_AKGC417L.txt', '104_1b1_Pr_sc_Litt3200.wav', '204_7p5_Al_mc_AKGC417L.wav', '112_1p1_Ll_sc_Litt3200.wav', '172_2b5_Tc_mc_AKGC417L.txt', '130_3b3_Ll_mc_AKGC417L.txt', '130_1p4_Pl_mc_AKGC417L.txt', '207_2b4_Pr_mc_AKGC417L.wav']
/content/icbhi_dataset/raw/respiratory_sound_database/Respiratory_Sound_Database
    ['filename_format.txt', 'patient_diagnosis.csv', 'filename_differences.txt']
/content/icbhi_dataset/raw/respiratory_sound_database/Respiratory_Sound_Database/audio_and_txt_files
    ['158_1b3_Ar_mc_LittC2SE.txt', '203_1p3_Tc_mc_AKGC417L.txt', '138_2p2_Al_mc_AKGC417L.txt', '104_1b1_Pr_sc_Lit

In [42]:
split_path = "/content/icbhi_dataset/raw/ICBHI_challenge_train_test.txt"
url = "https://bhichallenge.med.auth.gr/sites/default/files/ICBHI_challenge_train_test.txt"

# Use curl with --insecure to bypass SSL certificate verification
!curl --insecure -L "{url}" -o "{split_path}"

print("Saved to:", split_path)
print("Exists:", os.path.exists(split_path))
print("Size:", os.path.getsize(split_path), "bytes")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   276  100   276    0     0     26      0  0:00:10  0:00:10 --:--:--    59
Saved to: /content/icbhi_dataset/raw/ICBHI_challenge_train_test.txt
Exists: True
Size: 276 bytes


In [54]:
master_df = build_master_annotation(CONFIG["raw_dir"], CONFIG["splits_file"])


--- Split file content (first 5 rows):
                                             filename  split
0  <!DOCTYPE html><html><head><title>404 Not Foun...    NaN

Found 920 annotation files...
  Debug: Stem '101_1b1_Al_sc_Meditron' resolved to split 'unknown'
  Debug: Stem '101_1b1_Pr_sc_Meditron' resolved to split 'unknown'
  Debug: Stem '102_1b1_Ar_sc_Meditron' resolved to split 'unknown'
  Debug: Stem '103_2b2_Ar_mc_LittC2SE' resolved to split 'unknown'
  Debug: Stem '104_1b1_Al_sc_Litt3200' resolved to split 'unknown'

Master DataFrame built.
Total breathing cycles : 6898
Unique patients        : 126
Train cycles           : 0
Test cycles            : 0


In [55]:
split_path = CONFIG["splits_file"]
with open(split_path, 'r') as f:
    content = f.read(500) # Read first 500 characters to inspect
print(f"Content of {split_path}:\n")
print(content)


Content of /content/icbhi_dataset/raw/ICBHI_challenge_train_test.txt:

<!DOCTYPE html><html><head><title>404 Not Found</title></head><body><h1>Not Found</h1><p>The requested URL "https://bhichallenge.med.auth.gr/sites/default/files/ICBHI_challenge_train_test.txt" was not found on this server.</p></body></html>SMTP Error: Could not authenticate.

